# Run experiments — XGBoost (piloto)

Notebook delgado para Colab. El codigo del pipeline vive en `src/tesis_forecast` (versionado en Git); este notebook solo monta Drive, instala dependencias, trae el codigo, define la configuracion del experimento y llama a `run_experiment(...)`.

## Datos de entrada

Los 34 archivos (`IGAE_2.xlsx`, `Temperaturas promedio.csv`, y por cada una de las 8 regiones -- BCA, CEN, NES, NOR, NTE, OCC, ORI, PEN -- `{REGION}_long.csv`, `{REGION}_GEN.csv`, `{REGION}_IMP.csv`, `{REGION}_EXP.csv`) viven permanentemente en Google Drive, en `MyDrive/Bases de datos Tesis` (ruta centralizada en la constante `DATA_DIR` mas abajo). No hace falta subirlos a mano en cada sesion.

Detalle exacto de columnas/formato esperado por archivo: [`docs/DATOS_REQUERIDOS.md`](../docs/DATOS_REQUERIDOS.md) en el repo.

## Setup (una sola celda): montar Drive, instalar dependencias, cargar el proyecto

Ajusta `REPO_URL` a la URL real del repositorio dedicado a este proyecto. Mientras no exista, sube la carpeta `src/tesis_forecast` a `/content/tesis_repo/src/tesis_forecast` por otro medio (por ejemplo subiendola a Drive y copiandola) y salta la linea de `git clone`.

In [ ]:
import os
import sys

# 1. Montar Google Drive (los resultados se guardan en
#    /content/drive/MyDrive/Pipeline_Resultados/<RUN_NAME>/).
#    run_experiment() tambien monta Drive por su cuenta si hace falta,
#    pero se monta aqui explicitamente para detectar problemas de acceso
#    antes de gastar tiempo de computo entrenando.
from google.colab import drive
drive.mount("/content/drive")

# 2. Instalar dependencias. pandas, numpy, scikit-learn y xgboost ya vienen
#    preinstalados en el runtime estandar de Colab; optuna no.
!pip install -q optuna

# 3. Traer el codigo del proyecto (Git).
REPO_URL = "https://github.com/CarlosT0503/Tesis-forecasting.git"
REPO_DIR = "/content/tesis_repo"

if REPO_URL == "REEMPLAZA_CON_LA_URL_DEL_REPO":
    print(
        "AVISO: REPO_URL todavia no esta configurado. "
        "Sube src/tesis_forecast a " + REPO_DIR + "/src/tesis_forecast "
        "por otro medio, o actualiza REPO_URL y vuelve a correr esta celda."
    )
elif not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("\nSetup completo.")
print("REPO_DIR:", REPO_DIR)
print("SRC_DIR en sys.path:", SRC_DIR in sys.path)

In [ ]:
from tesis_forecast.config import ExperimentConfig
from tesis_forecast.runner import run_experiment

In [ ]:
# Ruta donde viven permanentemente los datos de entrada en Google Drive.
# Centralizada aqui (en vez de en el default de run_experiment()) para que
# el paquete siga siendo portable entre Colab, local u otros entornos; un
# futuro matrix runner puede reutilizar esta misma constante para todas
# las corridas en vez de repetirla por config.
DATA_DIR = "/content/drive/MyDrive/Bases de datos Tesis"

## Configuracion del experimento piloto

Config exactamente igual a la vigente en el notebook legacy (celda 49): train 336h, forecast 168h, las 8 exogenas, 10 trials de Optuna. Dejar los campos en `None` usa esos mismos defaults; se muestran explicitos aqui solo para que quede claro cual es la config piloto.

In [ ]:
config = ExperimentConfig(
    modelo="xgboost",
    exogenas=[
        "Temperatura",
        "Primarias",
        "Secundarias",
        "Terciarias",
        "IGAE",
        "Generacion",
        "Importacion",
        "Exportacion",
    ],
    train_hours=336,
    forecast_horizon=168,
    optuna_n_trials=10,
    notas="Piloto de migracion: config identica a la vigente en el notebook legacy (celda 49).",
)

# run_experiment() devuelve un ExperimentResult (run_name, run_dir, status,
# regiones_esperadas, regiones_con_metricas, mape_promedio, archivos).
# Si algo falla, levanta una excepcion en vez de devolver un resultado con
# status="failed" -- revisa run_dir/log.txt para el traceback completo.
resultado = run_experiment(config, data_dir=DATA_DIR, mount_drive=True)
resultado

## Matriz de exogenas individuales (una exogena a la vez)

Corre, para cada modelo **multivariado** (el que tenga al menos una exogena en su catalogo -- ver `runner.MODEL_DEFAULTS`), una corrida por cada exogena de `EXOGENAS_INDIVIDUALES`, **sin acumular** (nunca dos exogenas juntas en la misma corrida). Los modelos univariados (hoy: `naive`, `naive_trend`, `ar`, `naive_trend_seasonal`, `ar_resid_trend_seasonal`) se excluyen automaticamente -- no generan corridas redundantes con una sola exogena c/u.

Cada corrida usa exactamente los defaults cientificos vigentes del modelo (`train_hours`/`forecast_horizon`/`optuna_n_trials`, ver `runner.MODEL_DEFAULTS`); lo UNICO que cambia respecto a la matriz baseline es `exogenas`. El `RUN_NAME` de cada corrida queda en una carpeta propia (ej. `XGBoost_train336h_fh168h_Temp` vs. `XGBoost_train336h_fh168h_IGAE` vs. el baseline `XGBoost_train336h_fh168h_Temp-Prim-Sec-Terc-IGAE-Gen-Imp-Exp`), asi que **no toca ni sobrescribe** los resultados que ya esta produciendo la matriz baseline en otro Colab. El checkpoint por region (`docs/CHECKPOINT_RESUME.md`) aplica exactamente igual: una corrida individual completa se salta, una incompleta se reanuda solo desde las regiones faltantes.

**Esta celda solo arma y muestra la lista de configs -- no lanza ningun experimento.** Ver `tests/test_individual_exog_matrix.py` para la verificacion (numero de configs, sin colisiones de RUN_NAME, defaults preservados, etc.).

In [ ]:
from tesis_forecast.matrix import build_individual_exog_matrix, run_matrix, resumen_dataframe

EXOGENAS_INDIVIDUALES = [
    "Temperatura",
    "IGAE",
    "Generacion",
    "Importacion",
    "Exportacion",
]

configs_individuales = build_individual_exog_matrix(exogenas=EXOGENAS_INDIVIDUALES)

print(f"Total de corridas: {len(configs_individuales)}")
print("\nPor modelo:")
for modelo in sorted({c.modelo for c in configs_individuales}):
    exogenas_de_modelo = sorted(c.exogenas[0] for c in configs_individuales if c.modelo == modelo)
    print(f"  {modelo}: {exogenas_de_modelo}")

configs_individuales

### Lanzar la matriz de exogenas individuales

Solo correr esta celda cuando quieras lanzar de verdad las corridas (puede tardar horas/dias segun cuantos modelos/exogenas queden pendientes). `mount_drive=True` porque esta puede ser la primera celda que toca Drive en esta sesion si se salto la celda de setup del piloto.

In [ ]:
resultados_individuales = run_matrix(
    configs_individuales,
    data_dir=DATA_DIR,
    mount_drive=True,
)

resumen_dataframe(resultados_individuales)